# Retrato bibliométrico de quatro institutos da USP — recorte de 2025

**SCC0252 — Visualização Computacional | Segunda etapa do projeto | 2026/2**

**Integrantes:**
* Daniela Costa da Silva (14613625)
* Isaac Ferreira Souza Santos (15637912)
* Thiago Oliveira dos Santos (13696220)  
* Anna Lucia Moro Lanzuolo (13864436)

**Institutos analisados:** ICMC, EESC, IFSC e IQSC.

> **Sobre o período analisado.**
* O ano **2025** identifica as listas de pesquisadores usadas nesta investigação. As métricas disponíveis (documentos, citações e índice h) são totais acumulados registrados nas consultas realizadas.

## 1. Contexto e perguntas de investigação

A ideia deste trabalho é observar como a produção científica registrada nos perfis dos pesquisadores se distribui entre departamentos de quatro institutos da USP em São Carlos. Nosso objetivo não é classificar áreas de conhecimento como melhores ou piores, mas identificar diferenças de distribuição e perfis que mereçam investigação.

O recorte é **transversal**, não longitudinal: não faremos comparações de crescimento, entrada ou saída de docentes entre anos. Mantivemos a estrutura de exploração do notebook anterior e organizamos cinco perguntas:

1. Como as citações se distribuem entre os departamentos de cada instituto?
2. Pesquisadores com mais documentos também tendem a registrar mais citações?
3. Quais pesquisadores se destacam em citações e em índice h?
4. As citações se concentram em uma parcela pequena dos pesquisadores?
5. Os departamentos apresentam perfis distintos de documentos, citações e índice h?

As perguntas são exploratórias. Citações e índice h dependem de área, tempo de carreira, práticas de publicação e cobertura do Google Scholar. As conclusões precisam respeitar essas diferenças.

## 2. Bibliotecas e arquivos

In [1]:
%pip -q install "dash>=2.17,<4" "plotly>=5.20" pandas numpy scipy

In [2]:
from pathlib import Path
import unicodedata
import math
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import spearmanr
from IPython.display import display, Markdown

ANO = 2025
ORDEM_INSTITUTOS = ["ICMC", "EESC", "IFSC", "IQSC"]
CORES = {"ICMC":"#244A73", "EESC":"#C07838", "IFSC":"#477D72", "IQSC":"#8A6496"}
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 65)


def formato_br(valor, casas=0):
    """Formatação numérica usada nos textos, independente do idioma do Colab."""
    return f"{valor:,.{casas}f}".replace(",", "§").replace(".", ",").replace("§", ".")

### 2.1. Padronização e controle de qualidade

Os arquivos usam cabeçalhos diferentes para a mesma informação — por exemplo, `documentos_total_consulta` no IQSC e `documentos total na consulta` nos demais. Antes de produzir gráficos, padronizamos os nomes das colunas, selecionamos **somente 2025**, removemos linhas completamente vazias e verificamos valores ausentes ou incoerentes.

A ausência de informação **não é tratada como zero**. Para comparações que envolvem documentos e citações, usamos somente os registros que possuem ambas as métricas. Para rankings de índice h, exigimos também um índice h válido.

In [3]:
FONTES = {
    "ICMC": ("DatasetICMC(1).csv", "https://raw.githubusercontent.com/Dani-ela-git/Visualization-Data/refs/heads/main/dataset.csv"),
    "EESC": ("DatasetEESC.csv", "https://raw.githubusercontent.com/Dani-ela-git/Visualization-Data/refs/heads/main/DatasetEESC.csv"),
    "IFSC": ("DatasetIFSC.csv", "https://raw.githubusercontent.com/Dani-ela-git/Visualization-Data/refs/heads/main/DatasetIFSC.csv"),
    "IQSC": ("DatasetIQSC - IQSC_dataset_2015_2025(1).csv", "https://raw.githubusercontent.com/Dani-ela-git/Visualization-Data/refs/heads/main/DatasetIQSC.csv"),
}

PASTAS_BUSCA = [Path.cwd(), Path("/content"), Path("/mnt/data"), Path.cwd() / "dados", Path("/content/dados")]


def normalizar_cabecalho(texto):
    texto = unicodedata.normalize("NFKD", str(texto))
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return " ".join(texto.strip().lower().replace("_", " ").split())


MAPA_COLUNAS = {
    "ano": "ano", "numero usp": "numero_usp", "nome": "nome",
    "nome do pesquisador": "nome",
    "identificador google scholar": "identificador_google_scholar",
    "identificador do google scholar": "identificador_google_scholar",
    "documentos total na consulta": "documentos_totais",
    "documentos total consulta": "documentos_totais",  # Variação do arquivo IQSC
    "documentos totais": "documentos_totais",
    "citacoes totais": "citacoes_totais", "citacoes total": "citacoes_totais",
    "indice h": "indice_h", "indice i10": "indice_i10",
    "departamento": "departamento", "tabela de origem": "departamento",
    "data consulta": "data_consulta",
}

COLUNAS_PADRAO = [
    "ano", "instituto", "departamento", "numero_usp", "nome",
    "identificador_google_scholar", "documentos_totais", "citacoes_totais",
    "indice_h", "indice_h_informado", "indice_i10", "data_consulta",
    "id_pesquisador", "alerta_qualidade", "registro_valido_documentos_citacoes"
]


def procurar_arquivo(nome):
    for pasta in PASTAS_BUSCA:
        caminho = pasta / nome
        if caminho.is_file():
            return caminho
    return None


def tratar_instituto(sigla, fonte):
    dados = fonte.copy().dropna(how="all")
    dados.columns = [normalizar_cabecalho(c) for c in dados.columns]
    dados = dados.rename(columns=MAPA_COLUNAS)
    obrigatorias = {"ano", "numero_usp", "nome", "departamento", "documentos_totais", "citacoes_totais"}
    faltantes = obrigatorias.difference(dados.columns)
    if faltantes:
        raise ValueError(f"{sigla}: faltam colunas obrigatórias: {sorted(faltantes)}")
    for coluna in ["identificador_google_scholar", "indice_h", "indice_i10", "data_consulta"]:
        if coluna not in dados.columns:
            dados[coluna] = pd.NA
    dados["ano"] = pd.to_numeric(dados["ano"], errors="coerce")
    dados = dados.loc[dados["ano"] == ANO].copy()
    dados["ano"] = ANO
    dados["instituto"] = sigla
    for coluna in ["numero_usp", "nome", "departamento", "identificador_google_scholar"]:
        dados[coluna] = dados[coluna].astype("string").str.strip().replace("", pd.NA)
    dados["departamento"] = dados["departamento"].str.upper()
    for coluna in ["documentos_totais", "citacoes_totais", "indice_h", "indice_i10"]:
        dados[coluna] = pd.to_numeric(dados[coluna], errors="coerce")
    dados["data_consulta"] = pd.to_datetime(dados["data_consulta"], errors="coerce").dt.strftime("%Y-%m-%d")

    # Mantemos a informação original para permitir auditoria, sem inventar correções.
    dados["indice_h_informado"] = dados["indice_h"]
    h_incoerente = ((dados["indice_h"] > dados["documentos_totais"]) & dados["documentos_totais"].notna()).fillna(False).astype(bool)
    valores_negativos = (dados[["documentos_totais", "citacoes_totais", "indice_h"]] < 0).any(axis=1)
    if valores_negativos.any():
        raise ValueError(f"{sigla}: há métricas negativas, que exigem revisão da fonte antes das análises.")
    dados.loc[h_incoerente, "indice_h"] = np.nan

    incompletos = dados[["documentos_totais", "citacoes_totais"]].isna().any(axis=1)
    avisos = np.select(
        [h_incoerente & incompletos, h_incoerente, incompletos],
        ["h_incoerente;metricas_incompletas", "h_incoerente", "metricas_incompletas"],
        default=""
    )
    dados["alerta_qualidade"] = avisos
    dados["registro_valido_documentos_citacoes"] = ~incompletos
    dados["id_pesquisador"] = dados["instituto"] + ":" + dados["numero_usp"].fillna("SEM_USP:" + dados["nome"].fillna("desconhecido"))

    if dados["numero_usp"].isna().any() or dados["nome"].isna().any() or dados["departamento"].isna().any():
        raise ValueError(f"{sigla}: existem docentes sem identificador, nome ou departamento.")
    duplicados = dados.duplicated("id_pesquisador", keep=False)
    if duplicados.any():
        raise ValueError(f"{sigla}: existem IDs repetidos em 2025; verificar as linhas antes de desduplicar.")
    return dados[COLUNAS_PADRAO].copy()

In [4]:
# Primeiro tenta os CSVs originais; se não estiverem disponíveis, usa a versão
# unificada já tratada. Só tenta os URLs do notebook anterior como último recurso.
originais_locais = {inst: procurar_arquivo(nome) for inst, (nome, _) in FONTES.items()}
arquivo_tratado = procurar_arquivo("Conjunto_2025_Tratado.csv")

if all(originais_locais.values()):
    frames = [tratar_instituto(inst, pd.read_csv(originais_locais[inst], dtype="string")) for inst in ORDEM_INSTITUTOS]
    dados = pd.concat(frames, ignore_index=True)
    procedencia = "Quatro CSVs originais enviados com o projeto"
elif arquivo_tratado is not None:
    dados = pd.read_csv(arquivo_tratado, dtype={"numero_usp": "string", "id_pesquisador": "string"})
    dados["data_consulta"] = dados["data_consulta"].astype("string")
    dados["registro_valido_documentos_citacoes"] = dados[["documentos_totais", "citacoes_totais"]].notna().all(axis=1)
    procedencia = "CSV unificado, previamente auditado a partir dos quatro originais"
else:
    try:
        frames = []
        for inst, (_, url) in FONTES.items():
            frames.append(tratar_instituto(inst, pd.read_csv(url, dtype="string")))
        dados = pd.concat(frames, ignore_index=True)
        procedencia = "Links públicos do repositório usados no trabalho original"
    except Exception as erro:
        raise FileNotFoundError(
            "Não encontrei todos os arquivos. Envie os 4 CSVs originais ou "
            "'Conjunto_2025_Tratado.csv' pelo painel Arquivos do Colab."
        ) from erro

assert set(dados["ano"].dropna().astype(int).unique()) == {2025}
assert dados["id_pesquisador"].is_unique
# Garantimos que arquivos prontos também preservem a política de qualidade do índice h.
invalidos_h = dados["indice_h"] > dados["documentos_totais"]
assert not invalidos_h.fillna(False).any(), "Há índices h incoerentes no arquivo tratado."

# Salva o resultado do pré-processamento: arquivo reproduzível para o restante do projeto.
dados.to_csv("Conjunto_2025_Tratado.csv", index=False, encoding="utf-8-sig")
validos = dados.loc[dados["registro_valido_documentos_citacoes"]].copy()

print(f"Fonte: {procedencia}")
print(f"Registros de 2025: {len(dados)} | Com documentos e citações: {len(validos)}")
print("Nota: totais acumulados obtidos em consultas de 2026.")

Fonte: Links públicos do repositório usados no trabalho original
Registros de 2025: 425 | Com documentos e citações: 411
Nota: totais acumulados obtidos em consultas de 2026.


/tmp/ipykernel_471/2736757331.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  dados = pd.concat(frames, ignore_index=True)


## 3. Visão geral da amostra

O número de pesquisadores varia bastante entre departamentos. Assim, uma soma grande de citações pode refletir tanto perfis muito citados quanto um departamento com mais docentes. Sempre que possível, observamos **a mediana por pesquisador** e o número de registros válidos, além das somas.

In [5]:
def resumo_departamentos(frame):
    return (
        frame.groupby(["instituto", "departamento"], as_index=False)
        .agg(pesquisadores=("id_pesquisador", "nunique"),
             soma_documentos=("documentos_totais", "sum"),
             soma_citacoes=("citacoes_totais", "sum"),
             mediana_documentos=("documentos_totais", "median"),
             mediana_citacoes=("citacoes_totais", "median"),
             mediana_h=("indice_h", "median"))
    )

resumo = resumo_departamentos(validos)
display(resumo.sort_values(["instituto", "mediana_citacoes"], ascending=[True, False]).round(1))

,instituto,departamento,pesquisadores,soma_documentos,soma_citacoes,mediana_documentos,mediana_citacoes,mediana_h
6,EESC,SHS,22,5164,71249,154.5,1876.0,23.0
3,EESC,SEP,18,3425,55327,169.5,1815.0,22.0
7,EESC,SMM,14,2257,29399,101.0,1778.0,22.0
1,EESC,SEL,36,7257,62094,169.0,1405.0,20.0
2,EESC,SEM,30,5161,45453,144.5,1246.5,21.0
5,EESC,SGS,8,1178,9035,120.5,1007.5,17.0
4,EESC,SET,21,4049,35688,116.0,765.0,16.0
0,EESC,SAA,13,1767,13802,109.0,627.0,16.0
8,EESC,STT,12,1939,11320,152.5,581.5,16.0
12,ICMC,SSC,22,4614,56156,205.0,2173.5,23.5


## 4. Análise 1 — As distribuições de citações são semelhantes entre departamentos?

Usamos **boxplots**, porque a média pode ser influenciada por pesquisadores com volumes de citações muito acima do restante. A caixa mostra o intervalo entre quartis e a linha interna indica a mediana; pontos afastados tornam a dispersão visível. A comparação faz sentido **dentro de cada instituto**, sem pressupor que todas as áreas têm a mesma cultura de citação.

In [6]:
def figura_boxplot(frame, instituto):
    d = frame[frame["instituto"] == instituto].copy()
    ordem = d.groupby("departamento")["citacoes_totais"].median().sort_values(ascending=False).index.tolist()
    fig = px.box(d, x="departamento", y="citacoes_totais", color="departamento",
                 category_orders={"departamento": ordem}, points="outliers",
                 hover_data=["nome", "documentos_totais", "indice_h"],
                 title=f"{instituto} · Distribuição de citações por departamento",
                 labels={"departamento": "Departamento", "citacoes_totais": "Citações acumuladas por pesquisador"})
    fig.update_layout(template="plotly_white", showlegend=False, height=445)
    return fig

# Exemplo com um instituto. O painel interativo, ao final, permite explorar os quatro.
figura_boxplot(validos, "ICMC").show()

## 5. Análise 2 — Há associação entre quantidade de documentos e citações?

Os pontos representam pesquisadores, não departamentos. A correlação de **Spearman** mede a associação monotônica entre as duas métricas sem exigir que a relação seja linear. Um valor positivo indica que perfis com mais documentos tendem a ter mais citações no conjunto observado; ele **não demonstra causalidade**. A existência de pontos isolados também merece atenção.

In [7]:
def tabela_correlacoes(frame):
    linhas = []
    for instituto, parte in frame.groupby("instituto", sort=False):
        pares = parte[["documentos_totais", "citacoes_totais"]].dropna()
        if len(pares) < 3 or pares.nunique().min() < 2:
            rho, p = np.nan, np.nan
        else:
            rho, p = spearmanr(pares["documentos_totais"], pares["citacoes_totais"])
        linhas.append({"instituto": instituto, "n": len(pares), "rho_spearman": rho, "p_valor": p})
    return pd.DataFrame(linhas).set_index("instituto").reindex(ORDEM_INSTITUTOS).reset_index()

correlacoes = tabela_correlacoes(validos)
correlacoes_exibicao = correlacoes.copy()
correlacoes_exibicao["rho_spearman"] = correlacoes_exibicao["rho_spearman"].map(lambda v: formato_br(v, 3))
correlacoes_exibicao["p_valor"] = correlacoes_exibicao["p_valor"].map(lambda v: f"{v:.2e}")
display(correlacoes_exibicao)
fig = px.scatter(validos[validos["instituto"] == "ICMC"],
                 x="documentos_totais", y="citacoes_totais", color="departamento",
                 hover_name="nome", hover_data=["indice_h"],
                 log_x=True, log_y=True, title="ICMC · Documentos e citações por pesquisador",
                 labels={"documentos_totais": "Documentos (escala log)",
                         "citacoes_totais": "Citações (escala log)"})
fig.update_traces(marker=dict(size=9, opacity=.83, line=dict(width=.6, color="white")))
fig.update_layout(template="plotly_white", height=500)
fig.show()

,instituto,n,rho_spearman,p_valor
0,ICMC,120,"0,873",1.19e-38
1,EESC,174,"0,830",1.94e-45
2,IFSC,72,"0,774",1.62e-15
3,IQSC,45,"0,784",1.87e-10


## 6. Análise 3 — Quem aparece entre os pesquisadores com maiores métricas?

Um ranking destaca casos individuais que os gráficos agregados não mostram. Mantivemos duas medidas distintas: **citações acumuladas** e **índice h**. Na segunda, não colocamos valores faltantes no final da lista como se fossem os maiores — problema que ocorria na versão anterior do notebook. Os rankings são descritivos e não equivalem a uma avaliação completa da qualidade de um pesquisador.

In [8]:
def ranking_pesquisadores(frame, instituto, metrica="citacoes_totais", n=10):
    if metrica not in ["citacoes_totais", "indice_h"]:
        raise ValueError("Métrica inválida")
    parte = frame[(frame["instituto"] == instituto) & frame[metrica].notna()].copy()
    ordem = [metrica, "citacoes_totais"] if metrica == "indice_h" else [metrica, "indice_h"]
    return parte.sort_values(ordem, ascending=False, na_position="last").head(n)[
        ["nome", "departamento", "documentos_totais", "citacoes_totais", "indice_h"]
    ]

for instituto in ORDEM_INSTITUTOS:
    print(f"\n{instituto} · Três maiores valores de citações até 2025")
    display(ranking_pesquisadores(validos, instituto, "citacoes_totais", 3).reset_index(drop=True))
print("\nEESC · Exemplo do ranking por índice h (apenas índices preenchidos)")
display(ranking_pesquisadores(validos, "EESC", "indice_h", 5).reset_index(drop=True))


ICMC · Três maiores valores de citações até 2025


,nome,departamento,documentos_totais,citacoes_totais,indice_h
0,André Carlos Ponce de Leon Ferreira de Carvalho,SME,1150,24240,78
1,Francisco Aparecido Rodrigues,SME,228,12217,42
2,Francisco Louzada Neto,SME,388,8475,42



EESC · Três maiores valores de citações até 2025


,nome,departamento,documentos_totais,citacoes_totais,indice_h
0,Marcelo Zaiat,SHS,791,17085,68
1,André Teófilo Beck,SET,396,12613,41
2,Maria Bernadete Amancio Varesche Silva,SHS,467,9133,53



IFSC · Três maiores valores de citações até 2025


,nome,departamento,documentos_totais,citacoes_totais,indice_h
0,Vanderlei Salvador Bagnato,FCM,1687,27821,84
1,Osvaldo Novais de Oliveira Junior,FCM,1106,26860,91
2,Luiz Vitor de Souza Filho,FCI,709,24821,80



IQSC · Três maiores valores de citações até 2025


,nome,departamento,documentos_totais,citacoes_totais,indice_h
0,Emanuel Carrilho,SQM,715,19758,62
1,Sergio Antonio Spinola Machado,SQF,870,14730,71
2,Juarez Lopes Ferreira da Silva,SQF,531,13054,56



EESC · Exemplo do ranking por índice h (apenas índices preenchidos)


,nome,departamento,documentos_totais,citacoes_totais,indice_h
0,Marcelo Zaiat,SHS,791,17085,68
1,Maria Bernadete Amancio Varesche Silva,SHS,467,9133,53
2,Gherhardt Ribatski,SEM,324,7193,45
3,Francisco Antonio Rocco Lahr,SET,1308,8846,44
4,Evaldo Luiz Gaeta Espindola,SHS,503,6967,42


## 7. Análise 4 — Qual é a concentração de citações?

A **Curva de Lorenz** compara a participação acumulada dos pesquisadores (ordenados do menos ao mais citado) com a fração acumulada de citações. Quanto mais a curva se afasta da diagonal, maior é a concentração. O **índice de Gini** resume a desigualdade em uma escala de 0 a 1.

Também calculamos que parcela das citações fica com os **10% de perfis mais citados** (arredondando a quantidade de pesquisadores para cima). É um retrato da distribuição de citações entre *perfis*, não de publicações únicas: em coautorias, citações podem aparecer em mais de um perfil.

In [9]:
def medidas_concentracao(valores):
    x = pd.to_numeric(pd.Series(valores), errors="coerce").dropna().to_numpy(dtype=float)
    x = np.sort(x[x >= 0])
    n = len(x)
    if n == 0 or x.sum() == 0:
        return {"n": n, "gini": np.nan, "top10_%": np.nan, "qtd_top10": 0,
                "x_lorenz": [0, 1], "y_lorenz": [0, 0]}
    proporcao_pessoas = np.arange(n + 1) / n
    proporcao_citacoes = np.insert(np.cumsum(x), 0, 0) / x.sum()
    gini = (2 * np.sum((np.arange(1, n + 1)) * x) / (n * x.sum()) - (n + 1) / n)
    qtd_top = math.ceil(n * 0.10)
    parte_top = 100 * x[-qtd_top:].sum() / x.sum()
    return {"n": n, "gini": float(gini), "top10_%": float(parte_top),
            "qtd_top10": qtd_top, "x_lorenz": proporcao_pessoas, "y_lorenz": proporcao_citacoes}

linhas = []
fig_lorenz = go.Figure()
fig_lorenz.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Igualdade perfeita",
                                line=dict(color="#8b929b", dash="dash")))
for inst in ORDEM_INSTITUTOS:
    m = medidas_concentracao(validos.loc[validos.instituto == inst, "citacoes_totais"])
    linhas.append({"instituto": inst, "pesquisadores": m["n"], "gini": m["gini"],
                   "quantidade_top10": m["qtd_top10"], "citações_dos_10%_mais_citados_%": m["top10_%"]})
    fig_lorenz.add_trace(go.Scatter(x=m["x_lorenz"], y=m["y_lorenz"], mode="lines",
                                    name=inst, line=dict(color=CORES[inst], width=3)))
concentracao = pd.DataFrame(linhas)
display(concentracao.round(3))
fig_lorenz.update_layout(title="Concentração de citações · Curvas de Lorenz dos quatro institutos",
                         xaxis_title="Participação acumulada de pesquisadores",
                         yaxis_title="Participação acumulada de citações",
                         xaxis_tickformat=".0%", yaxis_tickformat=".0%",
                         template="plotly_white", height=470, legend_title="Instituto")
fig_lorenz.show()

,instituto,pesquisadores,gini,quantidade_top10,citações_dos_10%_mais_citados_%
0,ICMC,120,0.617,12,43.233
1,EESC,174,0.547,18,40.692
2,IFSC,72,0.536,8,43.262
3,IQSC,45,0.469,5,32.976


## 8. Análise 5 — Os departamentos têm perfis bibliométricos diferentes?

Para ampliar a discussão, propomos uma **matriz de calor** que combina as medianas de documentos, citações e índice h por departamento. Como as três métricas têm escalas muito diferentes, a intensidade da cor não representa valores absolutos: aplicamos `log(1 + mediana)` e padronizamos **cada métrica dentro do instituto**. Tons mais fortes indicam posição relativamente mais alta naquela métrica.

No painel interativo, o mouse mostra a mediana original correspondente. Essa visualização permite observar departamentos que se destacam relativamente em uma dimensão e não em outra, sem transformar valores heterogêneos em uma pontuação de qualidade científica.

In [10]:
def figura_calor(frame, titulo="Perfil relativo por departamento"):
    if frame.empty:
        return go.Figure().update_layout(title="Sem dados para o filtro atual", template="plotly_white")
    agrupado = frame.groupby("departamento").agg(
        documentos=("documentos_totais", "median"),
        citacoes=("citacoes_totais", "median"),
        indice_h=("indice_h", "median"))
    nomes = ["Mediana de documentos", "Mediana de citações", "Mediana do índice h"]
    metricas = ["documentos", "citacoes", "indice_h"]
    originais = agrupado[metricas].T.to_numpy(dtype=float)
    log_originais = np.log1p(originais)
    escores = np.zeros_like(log_originais)
    for i, linha in enumerate(log_originais):
        validas = linha[np.isfinite(linha)]
        if not len(validas):
            escores[i] = np.nan
        elif np.std(validas) < 1e-12:
            escores[i] = np.where(np.isfinite(linha), 0.0, np.nan)
        else:
            escores[i] = (linha - np.mean(validas)) / np.std(validas)
    fig = go.Figure(go.Heatmap(
        x=agrupado.index.tolist(), y=nomes, z=escores,
        customdata=originais, zmid=0, colorscale="RdBu", reversescale=True,
        colorbar=dict(title="Escore relativo"),
        hovertemplate="%{x}<br>%{y}: %{customdata:,.1f}<br>Posição relativa: %{z:.2f}<extra></extra>"
    ))
    fig.update_layout(title=titulo, template="plotly_white", height=340,
                      margin=dict(l=185, r=30, t=60, b=50), xaxis_title="Departamento")
    return fig

figura_calor(validos[validos["instituto"] == "ICMC"], "ICMC · Perfil relativo das medianas").show()

## 9. Primeiras interpretações do recorte de 2025

Os números abaixo são calculados a partir dos arquivos; assim, o texto não depende de uma estimativa visual dos gráficos.

In [11]:
for instituto in ORDEM_INSTITUTOS:
    departamento_lider = (resumo[resumo["instituto"] == instituto]
                          .sort_values("mediana_citacoes", ascending=False).iloc[0])
    indicadores = concentracao.loc[concentracao["instituto"] == instituto].iloc[0]
    print(f"{instituto}: a maior mediana de citações aparece em {departamento_lider['departamento']} "
          f"({formato_br(departamento_lider['mediana_citacoes'], 1)} por perfil; "
          f"n={int(departamento_lider['pesquisadores'])}). "
          f"Gini={formato_br(indicadores['gini'], 3)}; "
          f"os {int(indicadores['quantidade_top10'])} perfis mais citados reúnem "
          f"{formato_br(indicadores['citações_dos_10%_mais_citados_%'], 1)}% das citações somadas.")

print("\nEsses resultados descrevem os registros disponíveis: não medem qualidade individual, "
      "não permitem inferir causalidade e não indicam crescimento anual.")

ICMC: a maior mediana de citações aparece em SSC (2.173,5 por perfil; n=22). Gini=0,617; os 12 perfis mais citados reúnem 43,2% das citações somadas.
EESC: a maior mediana de citações aparece em SHS (1.876,0 por perfil; n=22). Gini=0,547; os 18 perfis mais citados reúnem 40,7% das citações somadas.
IFSC: a maior mediana de citações aparece em FCM (2.541,0 por perfil; n=36). Gini=0,536; os 8 perfis mais citados reúnem 43,3% das citações somadas.
IQSC: a maior mediana de citações aparece em SQF (5.474,5 por perfil; n=24). Gini=0,469; os 5 perfis mais citados reúnem 33,0% das citações somadas.

Esses resultados descrevem os registros disponíveis: não medem qualidade individual, não permitem inferir causalidade e não indicam crescimento anual.


### Leitura crítica dos resultados

O **ICMC** ilustra por que não usamos somente somas: o SME reúne a maior soma de citações dos perfis válidos (87.582), mas a maior **mediana por pesquisador** é do SSC (2.173,5). O total depende, em parte, do número de perfis incluídos em cada departamento.

Nas quatro amostras, a associação de Spearman entre documentos e citações é positiva (aproximadamente de 0,77 a 0,87). Isso sustenta a ideia de que as duas métricas caminham juntas **nos registros estudados**, sem dizer que uma causa a outra. A presença de pesquisadores com muitos documentos e citações relativamente menores também ajuda a evitar conclusões simples demais.

Quanto à concentração, o ICMC apresenta Gini de aproximadamente **0,617** e o IQSC de **0,469**. No ICMC, os 12 perfis mais citados, correspondentes a 10% dos registros válidos, respondem por cerca de **43,2%** da soma das citações. Esses números tornam visível a desigualdade entre perfis, mas **não autorizam afirmar** que um instituto ou área faz pesquisa de maior qualidade do que outro.

## 10. Painel interativo com visualizações coordenadas

Esta parte atende ao objetivo da segunda etapa: além de permitir examinar os gráficos separadamente, as visualizações **conversam entre si**.

**Mecanismo 1 — consulta dinâmica:** altere o instituto, os departamentos, o mínimo de documentos e a métrica do ranking. Todas as visualizações se atualizam em conjunto.

**Mecanismo 2 — seleção com brushing e linking:** use a ferramenta de laço ou seleção retangular no gráfico de dispersão. A seleção é realçada nele e **filtra o boxplot, o ranking, a Curva de Lorenz e a matriz de calor simultaneamente**. O botão *Limpar seleção* restaura o conjunto filtrado pelos controles.

As duas novas visualizações coordenadas são **a Curva de Lorenz** e **a matriz de calor multidimensional**. Ambas acrescentam possibilidades de investigação que não estavam implementadas de forma equivalente no notebook anterior.

> Ao selecionar apenas alguns pesquisadores, os indicadores passam a se referir **à seleção**, não ao instituto inteiro. Isso é proposital: permite comparar grupos de perfis, mas requer cuidado para não generalizar resultados de seleções pequenas.

In [12]:
# Figuras utilizadas pelas callbacks do painel. São funções separadas para
# facilitar leitura, manutenção e teste.
def mensagem_vazia(texto):
    fig = go.Figure()
    fig.add_annotation(text=texto, x=.5, y=.5, xref="paper", yref="paper",
                       showarrow=False, font=dict(size=15, color="#596777"))
    fig.update_layout(template="plotly_white", xaxis=dict(visible=False),
                      yaxis=dict(visible=False), height=380)
    return fig


def grafico_dispersao(frame, selecionados=None):
    if frame.empty:
        return mensagem_vazia("Nenhum pesquisador atende aos filtros atuais.")
    df = frame[(frame["documentos_totais"] > 0) & (frame["citacoes_totais"] > 0)].copy()
    if df.empty:
        return mensagem_vazia("Não há valores positivos para a escala logarítmica.")
    fig = px.scatter(df, x="documentos_totais", y="citacoes_totais",
                     color="departamento", custom_data=["id_pesquisador", "nome", "indice_h"],
                     hover_name="nome", hover_data={"indice_h": True},
                     log_x=True, log_y=True,
                     labels={"documentos_totais": "Documentos (log)",
                             "citacoes_totais": "Citações (log)", "departamento": "Departamento"})
    fig.update_traces(marker=dict(size=10, line=dict(color="white", width=0.75)))
    if selecionados:
        ids = set(selecionados)
        for trace in fig.data:
            trace.selectedpoints = [i for i, row in enumerate(trace.customdata) if str(row[0]) in ids]
            trace.selected = dict(marker=dict(opacity=1, size=11))
            trace.unselected = dict(marker=dict(opacity=.15))
    fig.update_layout(template="plotly_white", height=475, dragmode="lasso",
                      margin=dict(l=50, r=20, t=35, b=50),
                      legend=dict(orientation="h", y=-.26),
                      uirevision="dispersao-estavel", selectionrevision="selecao-estavel")
    return fig


def grafico_boxplot(frame):
    if frame.empty: return mensagem_vazia("Sem dados na seleção.")
    ordem = frame.groupby("departamento")["citacoes_totais"].median().sort_values(ascending=False).index.tolist()
    fig = px.box(frame, x="departamento", y="citacoes_totais", color="departamento",
                 category_orders={"departamento": ordem}, points="outliers",
                 hover_data=["nome", "documentos_totais"],
                 labels={"departamento": "Departamento", "citacoes_totais": "Citações"})
    fig.update_layout(template="plotly_white", height=400, showlegend=False,
                      margin=dict(l=60, r=20, t=25, b=50))
    return fig


def grafico_lorenz(frame):
    info = medidas_concentracao(frame["citacoes_totais"])
    if not info["n"] or np.isnan(info["gini"]):
        return mensagem_vazia("Não há citações suficientes para calcular a concentração.")
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], name="Igualdade perfeita",
                             line=dict(color="#909aa6", dash="dash")))
    fig.add_trace(go.Scatter(x=info["x_lorenz"], y=info["y_lorenz"], mode="lines",
                             name=f"Seleção · Gini {info['gini']:.3f}",
                             line=dict(color="#244A73", width=3), fill="tonexty",
                             fillcolor="rgba(36,74,115,.09)"))
    fig.update_layout(template="plotly_white", height=400,
                      xaxis_title="Pesquisadores acumulados", yaxis_title="Citações acumuladas",
                      xaxis_tickformat=".0%", yaxis_tickformat=".0%",
                      margin=dict(l=50, r=20, t=25, b=60),
                      legend=dict(orientation="h", y=-.3))
    return fig


def grafico_ranking(frame, metrica):
    d = frame.dropna(subset=[metrica]).copy()
    if d.empty: return mensagem_vazia("Nenhum valor disponível para esse ranking.")
    d = d.sort_values([metrica, "citacoes_totais"], ascending=False).head(10).sort_values(metrica)
    fig = go.Figure(go.Bar(x=d[metrica], y=d["nome"], orientation="h",
                           marker_color="#477D72" if metrica == "indice_h" else "#244A73",
                           customdata=d[["departamento", "documentos_totais"]],
                           hovertemplate="%{y}<br>Valor: %{x:,.0f}<br>Departamento: %{customdata[0]}<extra></extra>"))
    fig.update_layout(template="plotly_white", height=400, margin=dict(l=235, r=25, t=30, b=50),
                      xaxis_title="Índice h" if metrica == "indice_h" else "Citações acumuladas")
    return fig

In [18]:

# @title PAINEL BIBLIOMÉTRICO USP — 2025
# @markdown ### Filtros das visualizações
# @markdown Altere os campos e clique em ▶ para atualizar os cinco gráficos.

Instituto = "Todos" # @param ["Todos", "ICMC", "EESC", "IFSC", "IQSC"]
Departamento = "Todos" # @param ["Todos", "SSC", "SCC", "SME", "SMA", "SAA", "SEL", "SEM", "SEP", "SET", "SGS", "SHS", "SMM", "STT", "FCM", "FCI", "SQF", "SQM"]
Minimo_documentos = 0 # @param {type:"slider", min:0, max:1000, step:25}
Ranking_por = "Citações" # @param ["Citações", "Índice h"]
Buscar_docentes = "" # @param {type:"string"}

import plotly.io as pio
from IPython.display import display, Markdown
import numpy as np

# Plotly diretamente na saída do Google Colab.
pio.renderers.default = "colab"

# Verificação das funções já existentes no notebook.
funcoes = [
    "grafico_dispersao",
    "grafico_boxplot",
    "grafico_lorenz",
    "figura_calor",
    "grafico_ranking",
    "medidas_concentracao"
]

faltantes = [f for f in funcoes if f not in globals()]

if faltantes:
    raise RuntimeError(
        "Execute primeiro as células que definem estas funções: "
        + ", ".join(faltantes)
    )

if "validos" not in globals():
    raise RuntimeError(
        "Execute primeiro a célula que cria o dataframe 'validos'."
    )

# --------------------------------------------------
# 1. Preparação dos dados
# --------------------------------------------------

base = validos.copy()

# Converter pd.NAType para np.nan nas colunas que vão para os gráficos
for col in base.columns:
    if base[col].dtype in ["boolean", "Int64", "string"] or hasattr(base[col], "cat"):
        base[col] = base[col].astype(object).where(base[col].notna(), None)

# Trabalhar exclusivamente com o recorte de 2025.
if "ano" in base.columns:
    base = base[
        pd.to_numeric(base["ano"], errors="coerce") == 2025
    ].copy()

for coluna in [
    "documentos_totais",
    "citacoes_totais",
    "indice_h"
]:
    base[coluna] = pd.to_numeric(
        base[coluna], errors="coerce"
    )

base = base.dropna(
    subset=["documentos_totais", "citacoes_totais"]
)

# --------------------------------------------------
# 2. Filtros dinâmicos
# --------------------------------------------------

if Instituto != "Todos":
    base = base[
        base["instituto"] == Instituto
    ]

if Departamento != "Todos":
    base = base[
        base["departamento"] == Departamento
    ]

base = base[
    base["documentos_totais"] >= Minimo_documentos
].copy()

# --------------------------------------------------
# 3. Seleção de pesquisadores
# --------------------------------------------------

# É possível buscar pelo nome ou número USP.
# Para selecionar vários, separe por vírgulas.

termos = [
    termo.strip().casefold()
    for termo in Buscar_docentes.split(",")
    if termo.strip()
]

if termos:
    nomes = base["nome"].fillna("").astype(str).str.casefold()
    numeros = base["numero_usp"].fillna("").astype(str)

    mascara = pd.Series(
        False, index=base.index
    )

    for termo in termos:
        mascara |= (
            nomes.str.contains(termo, regex=False)
            | numeros.str.contains(termo, regex=False)
        )

    selecionados = base.loc[mascara].copy()
else:
    selecionados = base.copy()

# Identificadores para destacar pontos no gráfico.
ids = (
    selecionados["id_pesquisador"]
    .astype(str)
    .tolist()
    if termos else None
)

# --------------------------------------------------
# 4. Indicadores do recorte
# --------------------------------------------------

total = len(selecionados)
soma_citacoes = selecionados["citacoes_totais"].sum()

concentracao = medidas_concentracao(
    selecionados["citacoes_totais"]
)

gini = concentracao["gini"]

gini_texto = (
    f"{gini:.3f}"
    if np.isfinite(gini)
    else "Indisponível"
)

display(Markdown(
    "## Retrato bibliométrico — USP 2025\n"
    "**Métricas acumuladas consultadas em 2026**"
))

print(f"Instituto: {Instituto}")
print(f"Departamento: {Departamento}")
print(f"Pesquisadores após filtros: {len(base)}")
print(f"Pesquisadores na seleção: {total}")
print(f"Soma de citações: {soma_citacoes:,.0f}")
print(f"Índice de Gini: {gini_texto}")

if termos and selecionados.empty:
    print(
        "\nNenhum pesquisador encontrado. "
        "Confira o nome, o número USP ou os filtros."
    )

# --------------------------------------------------
# 5. Gráfico de dispersão
# --------------------------------------------------

display(Markdown(
    "### 1. Relação entre documentos e citações"
))

fig1 = grafico_dispersao(
    base,
    selecionados=ids
)

fig1.show(renderer="colab")

# --------------------------------------------------
# 6. Boxplot
# --------------------------------------------------

display(Markdown(
    "### 2. Distribuição de citações por departamento"
))

fig2 = grafico_boxplot(selecionados)
fig2.show(renderer="colab")

# --------------------------------------------------
# 7. Curva de Lorenz
# --------------------------------------------------

display(Markdown(
    "### 3. Concentração de citações — Curva de Lorenz"
))

fig3 = grafico_lorenz(selecionados)
fig3.show(renderer="colab")

# --------------------------------------------------
# 8. Mapa de calor
# --------------------------------------------------

display(Markdown(
    "### 4. Perfil bibliométrico dos departamentos"
))

fig4 = figura_calor(
    selecionados,
    "Comparação das medianas dos departamentos"
)

fig4.show(renderer="colab")

# --------------------------------------------------
# 9. Ranking
# --------------------------------------------------

metrica = (
    "indice_h"
    if Ranking_por == "Índice h"
    else "citacoes_totais"
)

display(Markdown(
    f"### 5. Ranking dos pesquisadores — {Ranking_por}"
))

fig5 = grafico_ranking(
    selecionados,
    metrica
)

fig5.show(renderer="colab")

# --------------------------------------------------
# 10. Tabela dos pesquisadores selecionados
# --------------------------------------------------

display(Markdown(
    "### Pesquisadores incluídos na análise"
))

colunas = [
    "instituto",
    "departamento",
    "numero_usp",
    "nome",
    "documentos_totais",
    "citacoes_totais",
    "indice_h"
]

tabela = (
    selecionados[colunas]
    .sort_values(
        "citacoes_totais",
        ascending=False
    )
    .reset_index(drop=True)
)

display(tabela.head(30))

print("\nPainel executado com sucesso.")


## Retrato bibliométrico — USP 2025
**Métricas acumuladas consultadas em 2026**

Instituto: Todos
Departamento: Todos
Pesquisadores após filtros: 411
Pesquisadores na seleção: 411
Soma de citações: 1,110,490
Índice de Gini: 0.590


### 1. Relação entre documentos e citações

### 2. Distribuição de citações por departamento

### 3. Concentração de citações — Curva de Lorenz

### 4. Perfil bibliométrico dos departamentos

### 5. Ranking dos pesquisadores — Citações

### Pesquisadores incluídos na análise

,instituto,departamento,numero_usp,nome,documentos_totais,citacoes_totais,indice_h
0,IFSC,FCM,79280,Vanderlei Salvador Bagnato,1687,27821,84.0
1,IFSC,FCM,68824,Osvaldo Novais de Oliveira Junior,1106,26860,91.0
2,IFSC,FCI,5277515,Luiz Vitor de Souza Filho,709,24821,80.0
3,ICMC,SME,94764,André Carlos Ponce de Leon Ferreira de Carvalho,1150,24240,78.0
4,IQSC,SQM,61019,Emanuel Carrilho,715,19758,62.0
5,IFSC,FCI,10903551,Aion da Escossia Melo Viana,282,17122,59.0
6,EESC,SHS,1532684,Marcelo Zaiat,791,17085,68.0
7,IFSC,FCM,2599282,Igor Polikarpov,962,16979,68.0
8,IQSC,SQF,1869425,Sergio Antonio Spinola Machado,870,14730,71.0
9,IQSC,SQF,996680,Juarez Lopes Ferreira da Silva,531,13054,56.0



Painel executado com sucesso.
